# Deteksi dan Counting Bungkus Produk Minuman Indonesia

Notebook ini menjalankan pipeline Colab untuk:

- mount Google Drive,
- generate dataset sintetis format YOLO dari 1 gambar per kelas,
- training YOLOv8n,
- menyiapkan artefak model untuk serving Streamlit via GitHub/Streamlit Cloud.

Target folder proyek:

`/content/drive/My Drive/Colab Notebooks/vidconv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/My Drive/Colab Notebooks/vidconv"

In [ ]:
!pip install -q "cryptography>=42,<44" "pyOpenSSL>=24.0.0,<=24.2.1"
!pip install -q -r requirements.txt --upgrade-strategy only-if-needed

from importlib.metadata import version
for pkg in ['cryptography', 'pyOpenSSL', 'streamlit-webrtc', 'aiortc', 'av']:
    print(pkg, version(pkg))

print('Install selesai. Jika Colab meminta restart runtime setelah downgrade dependency, restart lalu lanjutkan dari cell berikutnya.')

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/My Drive/Colab Notebooks/vidconv')
DATASET = ROOT / 'dataset'

print('Root:', ROOT)
print('Dataset exists:', DATASET.exists())
for folder in sorted(DATASET.iterdir()):
    if folder.is_dir():
        print('-', folder.name, list(p.name for p in folder.iterdir()))

## Generate Dataset Sintetis YOLO

Gunakan `--samples-per-class 100` sampai `300`. Untuk training awal cepat, pakai 100. Untuk hasil lebih baik, pakai 200 atau 300.

In [ ]:
!python -m src.dataset_generator --samples-per-class 200 --image-size 640

In [ ]:
!cat generated_dataset/dataset.yaml
!find generated_dataset/images/train -type f | head
!find generated_dataset/labels/train -type f | head

## Preview Dataset Sintetis

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

image_files = list((ROOT / 'generated_dataset/images/train').glob('*.jpg'))
sample = random.choice(image_files)
label = ROOT / 'generated_dataset/labels/train' / f'{sample.stem}.txt'

img = cv2.imread(str(sample))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

for line in label.read_text().strip().splitlines():
    cls, xc, yc, bw, bh = map(float, line.split())
    x1 = int((xc - bw / 2) * w)
    y1 = int((yc - bh / 2) * h)
    x2 = int((xc + bw / 2) * w)
    y2 = int((yc + bh / 2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title(sample.name)
plt.show()

## Training YOLOv8n

Aktifkan GPU Colab: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!nvidia-smi

In [ ]:
!python train_yolo.py --data generated_dataset/dataset.yaml --epochs 80 --batch 8 --model yolov8n.pt

In [ ]:
from ultralytics import YOLO

model_path = ROOT / 'runs/beverage_yolov8n/weights/best.pt'
model = YOLO(str(model_path))
metrics = model.val(data=str(ROOT / 'generated_dataset/dataset.yaml'))
print('Model:', model_path)

## Siapkan Model untuk Streamlit via GitHub

Streamlit **tidak dijalankan dari Colab sebagai tahap akhir**. Colab dipakai untuk training dan menyiapkan file model. Serving dilakukan dari GitHub melalui Streamlit Cloud.

Cell berikut menyalin model hasil training ke `models/best.pt`, yaitu path default yang dipakai `app.py` saat deploy di Streamlit Cloud.

In [ ]:
!mkdir -p models
!cp runs/beverage_yolov8n/weights/best.pt models/best.pt
!ls -lh models/best.pt

## File yang Perlu Masuk GitHub

Push project ini ke GitHub setelah training selesai. File/folder penting untuk deployment:

- `app.py`
- `src/`
- `requirements.txt`
- `.streamlit/config.toml`
- `models/best.pt`
- `dataset/` opsional, hanya untuk nama kelas dan dokumentasi contoh data

Folder besar seperti `generated_dataset/` dan `runs/` tidak perlu masuk GitHub.

In [ ]:
!git status --short || true
!find models -maxdepth 1 -type f -print

## Deploy Streamlit via GitHub

Langkah deployment:

1. Buat repository GitHub, misalnya `vidconv-streamlit`.
2. Upload/push file project ke repository itu.
3. Pastikan `models/best.pt` ikut masuk repository. Jika ukuran model melebihi batas GitHub, gunakan Git LFS atau hosting model eksternal.
4. Buka Streamlit Cloud.
5. Pilih repository GitHub.
6. Isi main file path: `app.py`.
7. Deploy.

Aplikasi sudah memakai mode default `Browser camera - GitHub/Streamlit Cloud`, sehingga kamera dibuka dari browser pengguna melalui WebRTC.

Catatan kamera:

- Untuk deploy GitHub/Streamlit Cloud, gunakan mode `Browser camera - GitHub/Streamlit Cloud`.
- Mode `OpenCV camera - local runtime` hanya untuk komputer lokal atau Colab local runtime.
- Hosted Colab tidak dipakai untuk serving final Streamlit.